# **1. What is this class?**

* ConfigReader is a utility class (utils package).
* Its job: read configuration values from config.properties file so you don’t hardcode values (like URLs, usernames, passwords) in your test code.
* It is static-based, so you can call methods directly without creating an object:

`ConfigReader.loadConfig("dev");
String baseUrl = ConfigReader.getProperty("base.url");`


# **2. Class Members**

`private static Properties properties;`

* Properties is a Java built-in class from java.util.
* It works like a HashMap<String, String> but designed for config files (.properties).
* Example config.properties:


`base.url=https://example.com
browser=chrome
timeout=10
`

Loaded into Properties →
{ base.url=..., browser=..., timeout=... }



# 3. loadConfig(String env)

`public static void loadConfig(String env)`


* Static method → belongs to the class, not an object.
* env parameter is there for future use (e.g., qa, dev, prod). In your current code, it’s unused, but you could extend it to load different config files:
    * config.qa.properties
    * config.dev.properties


# 4. InputStream and ClassLoader

`InputStream input = ConfigReader.class.getClassLoader()
        .getResourceAsStream("config/config.properties")`


* InputStream → Java’s abstraction for reading data (file, network, memory).
* getResourceAsStream() → loads a file from the classpath (the compiled resources/ folder).
* Instead of hardcoding paths (src/test/...), this works universally.
* When you mvn test, Maven copies files from src/test/resources into target/test-classes/config/.
* ClassLoader looks inside target/test-classes automatically.
* So this line is where the config file comes from.


# **5. Null check**

`if(input==null) {
    throw new RuntimeException("❌ config.properties file not found in classpath!");
}
`

* If the file is missing, getResourceAsStream returns null.
* Good practice: fail fast with a clear error message.


# 6. Properties Loading

`properties = new Properties();
properties.load(input);`

load() reads the config.properties and puts key-value pairs into properties.


# 7. Error Handling

`catch (IOException e) {
    throw new RuntimeException("❌ Failed to load config.properties file: " + e.getMessage());
}`


* Any IOException (like corrupted file) will stop the framework with a clear message.
* Interview point: This is called Exception Wrapping (wrapping checked exception into unchecked RuntimeException).


# 8. getProperty(String key)

`public static String getProperty(String key)`

* Reads a value from the properties object.
* Example:
`
String url = ConfigReader.getProperty("base.url");`


Before returning, it checks if properties is loaded:

`if (properties == null) {
    throw new RuntimeException("⚠️ Config file not loaded! Call loadConfig() before using getProperty().");
}
`

→ prevents NullPointerException if loadConfig() wasn’t called.

## Interview Points You Can Say

1. Why use ConfigReader?

* To externalize configuration (base URLs, credentials, timeouts) instead of hardcoding them.
* Makes framework flexible: Just change config.properties, no code change needed.

2. Why use Properties class?
* Java provides Properties in java.util specifically for config files.
* Easy to load .properties format (key=value pairs).
* Provides methods like .getProperty(), .setProperty().

3. Why use ClassLoader.getResourceAsStream()?
* To load files from classpath (so it works in Maven/Gradle build).
* Avoids OS-dependent hardcoded paths like C:\....

4. Why static methods and variables?
* Config is global; you don’t want multiple instances of it.
* static ensures the same config is shared across the entire framework.

5. What if multiple environments exist (dev/qa/prod)?
Pass env to decide which config file to load:
`
java
getResourceAsStream("config/config-" + env + ".properties")`

`sh
mvn test -Denv=qa`

6. What if property key doesn’t exist?
* getProperty() will return null.
* You can enhance by adding a default value:

`return properties.getProperty(key, "default");`

**⚡ In short, explain to the interviewer:

#### "ConfigReader is a utility class that loads configuration values from a .properties file using Java’s built-in Properties API. It uses ClassLoader.getResourceAsStream() to fetch files from the classpath, which makes it environment-independent and portable. We keep methods static since config values are globally shared across the framework. This helps us externalize environment data and avoid hardcoding values into test scripts."**

